In [2]:
# =========================
# MODEL INITIALIZATION
# =========================
# This cell initializes the neural network architecture used for inference.
#
# The architecture is reconstructed so that the pretrained weights of the
# best-performing model can later be loaded for prediction.
#
# At the end of the cell, TensorFlow prints:
# - The model summary
# - The total number of trainable parameters
#
# Subsequent sections of this notebook will:
# - Load the trained model weights
# - Generate next-token predictions
# - Display qualitative prediction examples
# - Estimate inference speed on CPU

import tensorflow as tf
import numpy as np

class DimensionScaling(tf.keras.layers.Layer):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def build(self, input_shape):
        self.alpha = self.add_weight(shape=(self.dim,),initializer="ones",trainable=True,name="alpha")
    def call(self, inputs):
        return inputs*self.alpha
class SaveEveryNEpochs(tf.keras.callbacks.Callback):
    def __init__(self, n):
        self.n = n

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.n == 0:
            self.model.save(f"model_epoch_{epoch+1}.keras")

def dataset_from_file(i):
    X = np.load(f"/kaggle/input/datasets/jdoesagazl/deterministic-model/deterministic_tf_input_{i:04d}.npy").astype(np.float32)
    Y = np.load(f"/kaggle/input/datasets/jdoesagazl/deterministic-model/deterministic_tf_output_{i:04d}.npy")
    Y = ((Y[:, None] >> np.arange(48)) & 1).astype(np.float32)
    return tf.data.Dataset.from_tensor_slices((X, Y))

print(tf.config.list_physical_devices('GPU'))

inputs = tf.keras.Input(shape=(350,)) # redundant in theory but not in practice
x = DimensionScaling(350)(inputs) # scale so that context and vectors have the same contribution

# Interaction block
x = tf.keras.layers.Dense(350, activation='relu')(x)
x = tf.keras.layers.Dense(350, activation='relu')(x)
x = tf.keras.layers.Dense(350, activation=None)(x)
x = tf.keras.layers.LayerNormalization()(x)
x = tf.keras.layers.Activation('gelu')(x)

x = tf.keras.layers.Dense(20<<8, activation='relu')(x) # 256 of dimension 20

for N in range(7, 1, -1):
    x = tf.keras.layers.Reshape((1<<N, 2, 20))(x)
    x1 = x[:, :, 0, :]   # (batch, groups, 20)
    x2 = x[:, :, 1, :]   # (batch, groups, 20)
    
    concat = tf.keras.layers.Concatenate()([x1, x2])

    h = tf.keras.layers.Dense(40, activation='relu')(concat)

    gate = tf.keras.layers.Dense(20, activation='sigmoid')(h)
    candidate = tf.keras.layers.Dense(20)(h)

    merged = gate * candidate

    x = tf.keras.layers.Activation('relu')(merged)


# Now we have 4 vectors of dimension 20 which I will reshape as 2 of dimension 40
x = tf.keras.layers.Reshape((2,40))(x)
x = tf.keras.layers.Dense(24, activation='relu')(x) # 2 of dimension 24 instead of 20
x = tf.keras.layers.Reshape((48,))(x)
outputs = tf.keras.layers.Dense(48, activation='sigmoid')(x)# output layer
model = tf.keras.Model(inputs, outputs)
model.summary()
model.count_params()

[]


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 350)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dimension_scaling_1 │ (None, 350)       │        350 │ input_layer_1[0]… │
│ (DimensionScaling)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 350)       │    122,850 │ dimension_scalin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 350)       │    122,850 │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 350)       │    122,850 │ dense_25[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 350)       │        700 │ dense_26[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 350)       │          0 │ layer_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 5120)      │  1,797,120 │ activation_7[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_8 (Reshape) │ (None, 128, 2,    │          0 │ dense_27[0][0]    │
│                     │ 20)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_12         │ (None, 128, 20)   │          0 │ reshape_8[0][0]   │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_13         │ (None, 128, 20)   │          0 │ reshape_8[0][0]   │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 128, 40)   │          0 │ get_item_12[0][0… │
│ (Concatenate)       │                   │            │ get_item_13[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 128, 40)   │      1,640 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 128, 20)   │        820 │ dense_28[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 128, 20)   │        820 │ dense_28[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_6          │ (None, 128, 20)   │          0 │ dense_29[0][0],   │
│ (Multiply)          │                   │            │ dense_30[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 128, 20)   │          0 │ multiply_6[0][0]  │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_9 (Reshape) │ (None, 64, 2, 20) │          0 │ activation_8[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_14         │ (None, 64, 20)    │          0 │ reshape_9[0][0]   │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,189,736 (8.35 MB)

 Trainable params: 2,189,736 (8.35 MB)

 Non-trainable params: 0 (0.00 B)

2189736

In [3]:
# =========================
# MODEL LOADING AND TOKEN PREDICTION
# =========================
# This cell loads:
# - The weights of the best-performing trained model
# - The alpha coefficients used in the weighted bitwise scoring function
# - The vocabulary codeword encodings corresponding to the reduced vocabulary
#
# The neural network predicts probabilities for each individual bit of the
# next-token codeword. Those probabilities are then combined through the
# scoring function described in the preprint to rank candidate token encodings.
#
# The top-10 highest-scoring token predictions are extracted and compared
# against the expected target codewords to determine prediction correctness.
#
# The file `vocabulary_encoding.npy` contains the codeword representations
# of the approximately 56k vocabulary tokens used during training.

path_example = "/kaggle/input/datasets/jdoesagazl/model-and-examples/"
model.load_weights(f"{path_example}deterministic_best_model.keras")
alpha = np.load(f"{path_example}deterministic_best_alpha.npy")
X = np.load(f"{path_example}deterministic_NN_input.npy")
Y = np.load(f"{path_example}deterministic_NN_output.npy")
preds = model.predict(X)

path_token ='/kaggle/input/datasets/jdoesagazl/token-encoding/'
encoding = np.load(f"{path_token}vocabulary_encoding.npy")
encoding_bits = ((encoding[:, None] >> np.arange(48)) & 1).astype(np.int8)
    
eps = 1e-6
preds = np.clip(preds, eps, 1 - eps)
    
logit = np.log(preds) - np.log(1 - preds)   # (N_preds, bits)
bias  = np.log(1 - preds)                  # (N_preds, bits)
 
# apply bit weights
logit *= alpha
bias  *= alpha
    
# main computation
scores = logit @ encoding_bits.T + bias.sum(axis=1, keepdims=True)
result = np.argsort(-scores, axis=1)
result = result[:,:10]
    
out = encoding[result]
matches = out==Y[:, None]
del encoding_bits

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step


In [4]:
# =========================
# QUALITATIVE PREDICTION EXAMPLES
# =========================
# This cell converts the predicted token codewords back into their
# corresponding text tokens and displays qualitative next-token
# prediction examples generated by the model.
#
# For each input phrase:
# - The model outputs the top-10 predicted tokens
# - Predictions matching the expected target token are highlighted in green
#
# The displayed examples are intended to provide an intuitive view of the
# types of semantic and linguistic patterns captured by the architecture.

from pickle import load
from colorama import Fore, Back, Style

with open(f"{path_token}tokens.pkl", 'rb') as f: tokens = load(f)
encoding = np.load(f"{path_token}encoding.npy")
map_token = {encoding[i]:tokens[i] for i in range(encoding.shape[0])}
output_strings = [[map_token[out[i][j]].lower() for j in range(out.shape[1])] for i in range(out.shape[0])]
with open(f"{path_example}deterministic_NN_phrases.pkl", 'rb') as f: phrases = load(f)

print("\n".join(
    f"{'='*100}\nInput: {phrases[i]}\n\nPredictions:\n" +
    "  -  ".join(
        f"{Back.GREEN}{Fore.BLACK}{Style.BRIGHT}{output_strings[i][j]}{Style.RESET_ALL}"
        if matches[i,j] else output_strings[i][j]
        for j in range(len(output_strings[i]))
    )
    for i in range(len(phrases))
))

Input: As your letter is _ decided _ , the scaffolding shall be re - erected round Charley 's boots ( it has been taken down , and the 

Predictions:
chap  -  sits  -  1903  -  bloke  -  greets  -  standing  -  sight  -  neighbour  -  32nd  -  16th
Input: I have tried to exercise it with the utmost delicacy and discretion , and to suggest to you , especially towards the end , how this sort of writing ( regard 

Predictions:
that  -  its  -  's  -  to  -  serves  -  12th  -  11:15  -  26th  -  23rd  -  12:45
Input: I 

Predictions:
have  -  would  -  know  -  did  -  wondering  -  could  -  concerned  -  am  -  thought  -  doubted
Input: The Champion came out with us into the passage 

Predictions:
,  -  .  -  19  -  12:45  -  17  -  28  -  11:15  -  11:10  -  26  -  767
Input: 

Predictions:
but  -  still  -  mentioned  -  wondering  -  i̇  -  though  -  changed  -  i̇t  -  know  -  speaking
Input: ' But is it really possible to please the world ! ' says 

Predictions:
the  -  edward  

In [5]:
# =========================
# CPU INFERENCE BENCHMARK
# =========================
# This cell estimates the inference speed of the architecture on a
# standard Kaggle CPU environment.
#
# The benchmark includes:
# - Neural network prediction of bit probabilities
# - Computation of the weighted bitwise scoring function
# - Ranking of token candidates through score sorting
#
# The procedure is repeated multiple times to obtain a stable timing
# estimate. The script also prints basic CPU information, including
# available vectorization instructions such as AVX and AVX2.
#
# The reported execution time corresponds to prediction and decoding
# of a batch containing 100 input samples.

import time
import os
path_example = "/kaggle/input/datasets/jdoesagazl/model-and-examples/"
model.load_weights(f"{path_example}deterministic_best_model.keras")
alpha = np.load(f"{path_example}deterministic_best_alpha.npy")
X = np.load(f"{path_example}deterministic_NN_input.npy")
Y = np.load(f"{path_example}deterministic_NN_output.npy")
path_token ='/kaggle/input/datasets/jdoesagazl/token-encoding/'
encoding = np.load(f"{path_token}vocabulary_encoding.npy")
encoding_bits = ((encoding[:, None] >> np.arange(48)) & 1).astype(np.int8)
eps = 1e-6
n_runs = 100

start = time.time()
for i in range(100):
    preds = model.predict(X, verbose=0)
    preds = np.clip(preds, eps, 1 - eps)
        
    logit = np.log(preds) - np.log(1 - preds)   # (N_preds, bits)
    bias  = np.log(1 - preds)                  # (N_preds, bits)
     
    # apply bit weights
    logit *= alpha
    bias  *= alpha
        
    # main computation
    scores = logit @ encoding_bits.T + bias.sum(axis=1, keepdims=True)
    result = np.argsort(-scores, axis=1)
end = time.time()
avg_time = (end-start)/n_runs

# Timing formatting
total_ms = avg_time * 1000
per_sample_ms = (avg_time / X.shape[0]) * 1000

# Extract only relevant CPU info
cpu_info = os.popen(
    "lscpu | grep -E 'Model name|CPU\\(s\\)|Thread\\(s\\) per core|Core\\(s\\) per socket'"
).read()
flags = os.popen("lscpu | grep 'Flags'").read().lower()
flags = flags.strip().split()
avx_info = [F for F in flags if 'avx' in F]
avx_str = ", ".join(avx_info) if avx_info else "N/A"
print(f"""
=== CPU Inference Benchmark ===

Total time (100 samples): {total_ms:.3f} ms
Time per sample:          {per_sample_ms:.3f} ms

--- CPU Info ---
{cpu_info.strip()}
Vectorization: {avx_str}
""")


=== CPU Inference Benchmark ===

Total time (100 samples): 396.584 ms
Time per sample:          3.966 ms

--- CPU Info ---
CPU(s):                                  4
On-line CPU(s) list:                     0-3
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      2
NUMA node0 CPU(s):                       0-3
Vectorization: avx, avx2

